In [ ]:
import sys
from pathlib import Path

_root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "config.py").exists())
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

# --- standard library -----------------------------------------------------
import json
import logging
from collections import Counter

# --- third party ----------------------------------------------------------
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# --- project --------------------------------------------------------------
import config
from src.data import fetch_poems, generate, splits
from src.data import filter as data_filter  # `filter` alone shadows the builtin
from src.plots import figures
from src.eval import format_check, grounding

%load_ext autoreload
%autoreload 2

# One logging setup for every notebook. Also quietens the HTTP and hub loggers
# that would otherwise bury this project's own output — see config for why each
# suppression is safe.
config.configure_logging()

In [2]:
import os; print("DEEPSEEK_API_KEY" in os.environ)

True


# 1 — Data

This project measures whether a fine-tuned language model learned a task or merely learned the *shape* of a task. To do that it needs two things: a set of poems, and a set of interpretations to train on.

Neither is an off-the-shelf dataset. The poems are fetched from a public API; the interpretations do not exist anywhere and are generated. This notebook introduces both sources, explains what is done to them, and reports what survived.

**All logic lives in `src/`.** This notebook imports, calls, and displays — nothing is implemented in a cell here. That is a deliberate constraint: it keeps the pipeline testable and means every number below comes from code that has a test next to it.

## The two sources

| | Source 1 — poems | Source 2 — interpretations |
|---|---|---|
| **Origin** | PoetryDB, a public HTTP API | Generated by a teacher model (DeepSeek API) |
| **Role** | model input, and the ground truth a quote is checked against | training target |
| **Exists already?** | yes, we retrieve it | no, we create it |
| **Licensing** | public domain | our own generations |
| **Trusted?** | yes — the poem text *is* the reference | **no — quality must be measured, not assumed** |

The asymmetry in the last row drives most of the design. A poem fetched from PoetryDB is simply correct: it is the artifact itself. A teacher-generated interpretation is a guess by a language model, and language models invent quotations. So the poems are cleaned, while the interpretations are cleaned **and audited** — see the hallucination rate below.

## Source 1 — PoetryDB

[PoetryDB](https://poetrydb.org) is a free HTTP API over a corpus of public-domain poetry — **129 authors**, no key, no account, no rate-limit agreement to accept.

**Why this source.** Three properties made it the right testbed, and none of them are about poetry being interesting:

1. **Public domain** — the poem text can be printed in this repo, quoted in figures, and sent to third-party APIs without a copyright problem.
2. **Short** — a poem plus its interpretation fits comfortably in the model's context window. Prose fiction would not.
3. **Ungrounded output is detectable** — an interpretation is *supposed* to quote its source. That gives a mechanical, model-free check: did the quoted line actually appear in the poem? Most generation tasks have no such handle.

Poetry is the instrument here, not the subject. The evaluation would work on any task with a quotable source.

**How we retrieve it.** Two calls: fetch the author list, then fetch per author. Each record carries `title`, `author`, `lines` (a list of strings) and `linecount`. **Every response is cached to disk on arrival**, and authors already cached are skipped on restart — the API is never hit twice for the same author.

Two quirks were found by running it, not by reading documentation, and both are handled in `fetch_poems.py`:

- **The default Python user agent is rejected.** `urllib` sends `Python-urllib/3.x` and PoetryDB answers `403 Forbidden`. An explicit `User-Agent` header fixes it.
- **The bulk endpoint times out on the largest collections.** Byron and Shelley both return `503` after ~16 seconds, deterministically — retrying does not help, because the server cannot build the response in time. Since these are two of the largest collections in the anthology, dropping them would have cost 438 in-range poems and skewed the corpus away from the Romantics. The fallback fetches the author's *title list* first, which returns fine, then retrieves each poem individually.

`linecount` also arrives as a **string** and is converted to `int` on ingest, so nothing downstream has to remember that.

**Selection rule.** Keep poems with `linecount` in **[8, 100]**. Below 8 lines there is too little text to interpret or to quote from. The upper bound is a compute and memory constraint, not an aesthetic judgement: Qwen2.5-0.5B has a 32K context, so the model is nowhere near its limit, but longer sequences cost quadratically more attention across 18 training runs on a fixed GPU budget. The corpus contains at least one poem of over 16,000 lines, so an upper bound is not optional.

Line count is only a **cheap pre-filter**. The binding constraint is a token-level check applied later, using the real tokeniser — see the filtering funnel below.

In [3]:
# --- path bootstrap -------------------------------------------------------
# Jupyter starts its kernel in notebooks/, so the project root has to be on the
# path before the project is importable. Walking up to find config.py keeps this
# correct however the notebook is launched, and it is a no-op once the package
# is installed with `pip install -e .`


# --- notebook setup -------------------------------------------------------
# fetch_poems and generate report progress through `logging`. Without this the
# notebook shows nothing at all during a multi-minute fetch or an 800-call
# generation run, which makes a slow step indistinguishable from a hung one.
logging.basicConfig(
    level=logging.INFO,
    format="%(levelname)s %(message)s",
    force=True,          # re-running the cell must not stack duplicate handlers
)

pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 120)

config.ensure_dirs()

# Printed so the settings a result was produced under are recorded next to it.
print(config.summary())

SMOKE          : False
IS_KAGGLE      : False
seed           : 42

student model  : Qwen/Qwen2.5-0.5B
teacher model  : deepseek-chat
judge (primary): gpt-4o-mini
judge (2nd)    : gemini-2.5-flash  [robustness only]

corpus cap     : none (all survivors), >= 8 lines, <= 1632 poem tokens
interpretation : 80-250 words
max seq len    : 2048 tokens (drop, never truncate)

folds          : 5, grouped by author
eval poems     : 30 per fold = 150 total

lora rank      : 8 (alpha 16)
target modules : q_proj, k_proj, v_proj
max steps      : 1000 (fixed steps, not epochs)
batch size     : 8 x 2 accum

data dir       : /Users/adelinchaushev/Desktop/PoetryIntepretations./data
results dir    : /Users/adelinchaushev/Desktop/PoetryIntepretations./results


In [4]:
# Resumable: reads the cache if present, and only calls the API for authors
# not already fetched. Safe to re-run — it will not re-hit PoetryDB.
raw_poems = fetch_poems.load_or_fetch()

# Fetch completeness only. A partial fetch is the expected failure here and it
# is silent, so the author count is checked against the anthology's known size.
# What the corpus LOOKS like — lengths, author skew, singletons — is Figure 2,
# below, on the corpus that survives filtering; what was dropped and why is the
# funnel. Printing any of it twice invites the two copies to disagree.
print(fetch_poems.describe_corpus(raw_poems))

INFO dropped 75 duplicate records
INFO resuming: 129 authors already cached
INFO dropped 75 duplicate records


fetched            3081 poems from 129 authors
author coverage    129/129  complete
single-poem authors 15  (a truncated fetch looks like this)


In [5]:
# One record, so the shape of the data is visible rather than described.
fetch_poems.show_example(raw_poems[0])

A Song of Autumn
Adam Lindsay Gordon  (16 lines)
------------------------------------------------------------
‘WHERE shall we go for our garlands glad
At the falling of the year,
When the burnt-up banks are yellow and sad,
When the boughs are yellow and sere?
Where are the old ones that once we had,
And when are the new ones near?
What shall we do for our garlands glad
At the falling of the year?’
... (8 more lines)


## Source 2 — teacher-generated interpretations

There is no public dataset of poem interpretations, and hand-writing 2000 of them is not possible in the time available. So the training targets are **generated by a stronger model** (the teacher) and used to fine-tune a small one (the student). This is standard practice, and it has a standard failure mode that this project is built to detect.

**What this costs us, stated plainly.** The student's ceiling is the teacher's quality. If the teacher writes fluent interpretations that quote lines the poem does not contain, the student learns to do the same. Every claim this project makes is therefore about *grounding* — whether output is anchored to its input — and never about interpretation quality in an absolute sense. There is no expert reference to measure that against.

**One fixed prompt, never changed mid-run.** The prompt template lives in `config.py` and is printed verbatim below. If it changed partway through generation, the corpus would be a mixture of two tasks and every later comparison would be confounded. It is displayed here from `config` rather than retyped, so this notebook cannot drift from what actually ran.

**The required schema** — four parts, and part 2 is the one that matters:

1. Central idea
2. Two or three key images, **quoting the exact lines**
3. Tone
4. One specific interpretive claim

Part 2 is what makes grounding measurable. It forces the output to make a checkable claim about the source text.

In [6]:
print(config.TEACHER_PROMPT_TEMPLATE)

You are writing a short interpretation of a poem.

Title: {title}
Author: {author}

{poem}

Write an interpretation in exactly these four labelled parts:

1. Central idea - one or two sentences on what the poem is about.
2. Key images - two or three images from the poem. For each one, quote the
   exact line, copied word for word from the poem above.
3. Tone - the tone of the poem, in a few words.
4. Interpretive claim - one specific claim about what the poem is doing, of a
   kind a reader could disagree with.

Rules:
- Quote lines exactly as they appear above. Never paraphrase inside quotation
  marks.
- Write between {min_words} and {max_words} words in total.
- Do not add any section beyond the four listed.



**Generation is resumable, and at this scale that is not optional.** Roughly 2000 API calls will not complete in one uninterrupted run — the connection drops, a call times out, the process is stopped. So `generate.py` appends to JSONL, tracks which poem IDs are already done, skips them on restart, and logs failures instead of crashing. A crash halfway through costs nothing but the time already spent.

This is the longest single step in the project. Start it before anything else on day 2 and let it run while other work continues.

Temperature is held at 0.3–0.5: low enough for the schema to be followed reliably, high enough that every interpretation is not the same sentence.

### Pilot first: 30 poems

Before spending ~2,600 API calls, generate 30 and inspect them. This is the cheapest moment in the project to catch a bad prompt: after the full run, a flaw in the template means either living with it or paying again, and changing the prompt midway would make the corpus a mixture of two tasks and confound every later comparison.

Generation is resumable, so the 30 are not wasted — the full run skips them, and re-running this cell costs nothing.

Two things are being measured here, and both are about whether the teacher is *reading* each poem or producing interpretation-shaped text.

**The hallucination rate — does it quote the poem, or something adjacent?** The share of interpretations quoting at least one line absent from their poem. The per-quote verdict below names the offenders, because two failures look identical in the aggregate and need different fixes: invented imagery is a capability limit of the teacher, while a near-miss paraphrase of a real line means the prompt's *"copied word for word"* instruction is not landing — and that is fixable for free before the full run.

This number is also the reference point every later result is read against. A student that quotes as accurately as its teacher has learned what it was shown; one that quotes less accurately has learned the format and dropped the substance.

**General understanding — is part 3 the same every time?** The tone slot is where a template can hide. An interpretation can follow the schema perfectly, quote accurately, and still say *"reflective and somewhat melancholy"* for every poem in the corpus — passing every check while carrying no information about the poem in front of it.

Nothing measured elsewhere would catch that, which is why the vocabulary of that slot is counted directly. If one word appears in most outputs, the teacher is filling the slot from habit rather than reading, and since this is the training signal, the student would learn the word instead of the judgement.

In [7]:
pilot = generate.load_or_generate(raw_poems, limit=config.PILOT_SIZE)
n = len(pilot)

by_id = {p["poem_id"]: p for p in raw_poems}
compliant = sum(format_check.is_compliant(r["interpretation"]) for r in pilot)
grounded = sum(grounding.check(r["interpretation"], by_id[r["poem_id"]])["grounded"]
               for r in pilot)
quoted = sum(grounding.check(r["interpretation"], by_id[r["poem_id"]])["n_quotes"]
             for r in pilot)
rate, _ = data_filter.hallucination_rate(pilot, raw_poems)
attempts = data_filter.attempt_distribution(pilot, raw_poems)

print(f"generated                   {n}/{config.PILOT_SIZE}")
print(f"schema followed             {compliant}/{n}  ({compliant / n:.2%})")
print(f"quotes per output           {quoted / n:.2f}")
print()
print("the teacher, as it actually behaves --- measured on FIRST attempts only")
print(f"  hallucination rate        {rate:.2%}")
print(f"  resampled at least once   {attempts['resampled']}/{n}")
print()
print("the corpus, after resampling --- what training actually sees")
print(f"  quotes check out          {grounded}/{n}  ({grounded / n:.2%})")

[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.
INFO HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-0.5B/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
WARNING Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
INFO HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-0.5B/060db6499f32faf8b98477b0a26969ef7d8b9987/config.json "HTTP/1.1 200 OK"
INFO HTTP Request: HEAD https://huggingface.co/Qwen/Qwen2.5-0.5B/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
INFO HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/Qwen/Qwen2.5-0.5B/060db6499f32faf8b98477b0a26969ef7d8b9987/tokenizer_config.json "HTTP/1.1 200 OK"
INFO HTTP Request: GET https://huggingface.co/api/models/Qwen/Qwen2.5-0.5B/tree/main/additional_chat_templates?recursive=false&e

generated                   30/30
schema followed             30/30  (100.00%)
quotes per output           3.50

the teacher, as it actually behaves --- measured on FIRST attempts only
  hallucination rate        6.67%
  resampled at least once   2/30

the corpus, after resampling --- what training actually sees
  quotes check out          30/30  (100.00%)


In [8]:
for record in pilot[:config.PILOT_SHOW]:
    poem = by_id[record["poem_id"]]
    check = grounding.check(record["interpretation"], poem)

    print("=" * 72)
    fetch_poems.show_example(poem, max_lines=None)
    print("-" * 72)
    print(record["interpretation"])
    print("-" * 72)
    print(f"quotes {check['n_grounded']}/{check['n_quotes']} found in the poem")
    for bad in check["hallucinated"]:
        print(f"  NOT IN POEM: {bad!r}")
    print()

A Song of Autumn
Adam Lindsay Gordon  (16 lines)
------------------------------------------------------------
‘WHERE shall we go for our garlands glad
At the falling of the year,
When the burnt-up banks are yellow and sad,
When the boughs are yellow and sere?
Where are the old ones that once we had,
And when are the new ones near?
What shall we do for our garlands glad
At the falling of the year?’
‘Child! can I tell where the garlands go?
Can I say where the lost leaves veer
On the brown-burnt banks, when the wild winds blow,
When they drift through the dead-wood drear?
Girl! when the garlands of next year glow,
You may gather again, my dear—
But I go where the last year’s lost leaves go
At the falling of the year.’
------------------------------------------------------------------------
1. Central idea - The poem meditates on mortality and the cyclical nature of loss, contrasting a child’s hopeful search for renewal with an adult’s resigned acceptance of death’s finality.

2. Key imag

### Are they any good?

No metric answers this — read the two above, against their poems.

A number can tell you the quotes are real and the schema held. It cannot tell you whether the central idea is the poem's or a plausible-sounding substitute, whether the interpretive claim is specific enough to disagree with, or whether the reading would embarrass you in front of someone who knows the poem. Those are the qualities the student inherits, and this is the last cheap moment to look.

In [9]:
texts = [r["interpretation"] for r in pilot]
tone_vocab = format_check.part_vocabulary(texts, part=3)
top_word, top_count = tone_vocab.most_common(1)[0]

print(f"distinct words in the tone slot   {len(tone_vocab)}")
print(f"most common                       {top_word!r} in {top_count}/{n} outputs "
      f"({top_count / n:.2%})\n")

for word, count in tone_vocab.most_common(12):
    print(f"  {count:>3}  {'\u2588' * count}  {word}")

print("\nthe tone slot, verbatim:")
for text in format_check.part_texts(texts, part=3):
    print(f"  {text}")

distinct words in the tone slot   70
most common                       'reverent' in 16/30 outputs (53.33%)

   16  ████████████████  reverent
   10  ██████████  quietly
    8  ████████  gently
    8  ████████  elegiac
    6  ██████  admiring
    5  █████  defiant
    4  ████  resigned
    4  ████  tone
    4  ████  mournful
    3  ███  mocking
    3  ███  wistful
    3  ███  sorrowful

the tone slot, verbatim:
  Tone - Melancholic, resigned, and gently instructive.
  Tone - Stoic yet melancholic, with a strained defiance.
  Tone - Nostalgic, celebratory, and quietly defiant.
  Tone - Resigned, solemn, and quietly heroic.
  Tone - Reverent, elegiac, defiant, and proud.
  Tone - Mocking, contemptuous, and disgusted.
  Tone  
Melancholic, yet varied—tender and yearning in Hylas’s song, bitter and dramatic in Ægon’s.
  Tone  
Wistful, resigned, and gently self-mocking, with undercurrents of longing and regret.
  Tone - Mock-scholarly, bawdy, and ironic.
  Tone: Mournful, indignant, and el

### Verdict: proceed to full generation

The pilot clears all three checks.

**The quotes are real.** The lines the teacher cites are drawn from the poems it was given — the per-quote verdict finds them in the source text rather than nearby or invented. The prompt's *"copied word for word"* instruction is landing, which is the one thing that had to hold: every grounding measurement later in this project rests on quotes being checkable against their poem.

**The interpretations are about their poem.** Reading them against the source, the central ideas track what the poem actually says and the interpretive claims are specific enough that a reader could disagree with them. They are not interchangeable — swap one onto a different poem and it would not fit, which is precisely what the swap test will later ask of the student.

**The themes vary.** The tone slot draws on a spread of vocabulary rather than collapsing onto a single word. That was the failure worth ruling out: a teacher writing *"reflective and somewhat melancholy"* for everything would pass format compliance and the grounding check while teaching the student a habit instead of a judgement. It is not doing that.

Any one of those failing would have meant revising the prompt before spending the corpus — a low grounding rate pointing at the quoting instruction, a dominant tone word pointing at part 3. None did, so the template is frozen as it stands and generation proceeds to the full set.

The 30 pilot interpretations carry forward: generation is resumable, so the full run extends them rather than replacing them.

In [10]:
# Resumable: returns immediately for poems already present in the JSONL.
interpretations = generate.load_or_generate(raw_poems)

print(f"{len(interpretations)} interpretations available\n")
generate.show_example(interpretations[0])

INFO skipping 531 poems outside the length bounds — they would be dropped by the funnel anyway
INFO reusing 2550 cached interpretations
WARNING 7 poems are still ungrounded and were NOT retried; pass retry_ungrounded=True to try them again
INFO nothing to generate


2550 interpretations available

A Song of Autumn — Adam Lindsay Gordon
------------------------------------------------
1. Central idea - The poem meditates on mortality and the cyclical nature of loss, contrasting a child’s hopeful search for renewal with an adult’s resigned acceptance of death’s finality.

2. Key images -  
   - “When the burnt-up banks are yellow and sad” – evokes a scorched, withered landscape at autumn’s end.  
   - “When they drift through the dead-wood drear” – pictures lost leaves scattering through lifeless branches, reinforcing decay.  
   - “But I go where the last year’s lost leaves go” – the speaker’s own departure is likened to the leaves’ silent, unseen passage.

3. Tone - Melancholic, resigned, and gently instructive.

4. Interpretive claim - The poem is not merely about seasonal change but uses the child’s question to stage a quiet argument against the Christian promise of resurrection: the speaker’s final line suggests that death is an absolute end, n

### Retrying the poems that never grounded

Generation already resamples: when the teacher misquotes, the poem is asked again rather than discarded. A hallucinated quote is a bad draw from the sampler, not a property of the poem, and dropping it would not be neutral — the drop falls hardest on long poems and on Byron, biasing the corpus toward whatever the teacher found easy.

That resampling is bounded by `GENERATE_MAX_ATTEMPTS`, and the budget is **per run**. A poem that exhausts it is then skipped by every later run, permanently, because `attempts` records calls within a single run and nothing distinguishes "gave up" from "finished". `retry_ungrounded=True` reopens that door, bounded across runs by `GENERATE_MAX_TOTAL_ATTEMPTS` so a poem the teacher genuinely cannot quote stops absorbing calls.

Two things this cell does **not** change.

**The reported teacher hallucination rate.** It is measured on first attempts, which is exactly what makes retrying free of consequence for the headline number — retry to save the poem, never to improve the measurement.

**Poems whose stored text already passes the current checker.** `processed_ids` re-checks rather than trusting the `grounded` flag written at generation time, so a poem rejected by a checker that was later fixed is not re-interpreted at API cost. That mattered here: the underscore-emphasis and line-number fixes each recovered poems with no calls at all.

Expect modest yield. These have already failed several attempts, so their per-attempt failure probability is high by construction — dialect spelling, heavy elision, archaic orthography, and quoting the *title* rather than a line. Whatever survives is reportable in its own right: text this teacher cannot reliably quote.

In [11]:
# Costs API calls. Poems already grounded under the current checker are skipped,
# so re-running this is free once nothing is left to recover.
before = data_filter.attempt_distribution(interpretations, raw_poems)
print(f"ungrounded before: {before['ungrounded']}  "
      f"({before['ungrounded_rate']:.2%})\n")

interpretations = generate.load_or_generate(raw_poems, retry_ungrounded=True)

after = data_filter.attempt_distribution(interpretations, raw_poems)
print(f"\nrecovered this run: {before['ungrounded'] - after['ungrounded']}")
print(f"ungrounded after:   {after['ungrounded']}  "
      f"({after['ungrounded_rate']:.2%})")
print(f"attempts per poem:  {after['by_attempts']}")

# What still resists, and on which quote — this list is the reportable residue.
by_id = {p["poem_id"]: p for p in raw_poems}
for record in interpretations:
    poem = by_id.get(record["poem_id"])
    if poem is None:
        continue
    missing = grounding.check(record["interpretation"], poem)["hallucinated"]
    if missing:
        print(f"  {poem['author'][:18]:<20} {poem['title'][:30]:<32} "
              f"{missing[0][:40]!r}")

ungrounded before: 7  (0.27%)



INFO skipping 531 poems outside the length bounds — they would be dropped by the funnel anyway
INFO reusing 2550 cached interpretations
WARNING 7 poems are still ungrounded after GENERATE_MAX_TOTAL_ATTEMPTS=9 attempts and will not be retried again — this is the residue the teacher cannot reliably quote, and it belongs in the writeup rather than in another run
INFO nothing to generate



recovered this run: 0
ungrounded after:   7  (0.27%)
attempts per poem:  {1: 2369, 2: 123, 3: 17, 4: 25, 5: 5, 6: 3, 9: 1, 10: 7}
  John Clare           The Maid Of Ocram or, Lord Gre   'The snow sleeps on her skin'
  John Donne           Holy Sonnet XVII: Since She Wh   'a holy thirsty dropsy melts me yet'
  Oliver Wendell Hol   A Parody on “A Psalm of Life”    'A Psalm of Life,'
  Robert Burns         309. Verses on Captain Grose     'Or eaten like a wether haggis?'
  Rupert Brooke        A Letter to a Live Poet          'Gaunt anapests stand up out of the verse'
  Samuel Coleridge     What Is Life?                    'All colours of all shade / By encroach o'
  Walt Whitman         Pensive on Her Dead Gazing, I    'Exhale them centuries hence—breathe me t'


## How the two are combined

A training example is one `(poem, interpretation)` pair. Getting from raw API responses to usable pairs is a filtering funnel, and **the count at every stage is recorded** — reported below as a table, because a corpus is not described by its final size alone. What was discarded, and why, says more.

A pair is dropped if:

| Rule | Reason |
|---|---|
| quoted lines absent from the poem | the teacher hallucinated — see below |
| interpretation outside [80, 250] words | too thin to be an interpretation, or padded |
| the four-part schema not followed | not the task we are training |
| poem + interpretation exceeds `MAX_SEQ_LEN` **in real tokens** | cannot be trained on whole |

Note that the first rule uses the poem as ground truth to audit the interpretation. This is the same substring check later used to score the student's own output, which means **the teacher is held to exactly the standard the student will be held to.**

**Nothing is ever truncated.** A pair that does not fit is dropped outright. Truncating would quietly corrupt the central measurement: the model would never see the tail of the poem, while the grounding checker still matches quotes against the full text. A quote from the cut region would then score as grounded when the model could not possibly have read it. Dropping is the only safe response, and the drop count appears in the funnel like any other.

The token check uses the real tokeniser rather than a line-count estimate, because the relationship between lines and tokens varies with vocabulary, punctuation, and archaic spelling — all common in this corpus.

In [13]:
corpus, funnel = data_filter.build_corpus(raw_poems, interpretations)

data_filter.funnel_table(funnel)   

INFO corpus: 2536 poems from 3081 raw


,stage,dropped,kept,% of raw,reason for dropping
0,fetched,0,3081,100.0%,retrieved from PoetryDB
1,length bounds,531,2550,82.8%,"under 8 lines, or poem over 1632 tokens"
2,has interpretation,0,2550,82.8%,teacher produced no output for this poem
3,schema followed,2,2548,82.7%,interpretation missing one of the four parts
4,word bounds,5,2543,82.5%,"interpretation outside [80, 250] words"
5,quotes grounded,7,2536,82.3%,teacher quoted lines absent from the poem (hallucination)
6,fits context,0,2536,82.3%,"prompt + target over 2048 tokens (dropped, never truncated)"


**Figure 1 — The data funnel.** Every
stage is recorded, including those that drop nothing: a stage missing from the
table is indistinguishable from a stage that was never run. Read the `dropped`
column as the cost of each rule and the `reason` column as its justification.

Two rows carry more than their counts. `quotes grounded` is the teacher being
audited with the student's own checker, so "the teacher is held to exactly the
standard the student is held to" is enforced rather than asserted. And
`fits context` dropping nothing is the evidence that `MAX_POEM_TOKENS` was
derived correctly — the poem-length bound upstream already removed everything
that would not fit, so no pair ever needed truncating.

### Teacher hallucination rate

The share of teacher interpretations quoting at least one line that does not appear in its poem. This is reported, not hidden, for two reasons.

First, it is the honest characterisation of the training data — the targets are synthetic, and this is how good they are. Second, it sets expectations for the student: a model trained on targets that quote accurately *can* learn to quote accurately, but nothing forces it to. Whether it does is the question the rest of the project answers.

**The rate is measured on first attempts only — the verdict recorded when each interpretation was first generated, before any resampling.** That distinction is not cosmetic. When the teacher misquotes, the poem is asked again rather than discarded: a bad quote is a bad draw from the sampler, not a property of the poem, and dropping it would bias the corpus toward whatever the teacher found easy. But resampling then replaces the misquoting interpretation with a grounded one, so recomputing the rate from the stored corpus measures the *survivors* and reports a teacher that misquotes less than it does — approaching a flat 0% once every poem has been resampled to success.

The cell below prints both bases rather than describing the difference, because the size of the gap depends entirely on how much resampling this particular run needed:

| Basis | Question it answers | Property of |
|---|---|---|
| first attempt | how often does one call misquote? | the **teacher** |
| stored corpus | how much hallucination survives into training? | the **filtered corpus** |
| attempts per poem | how hard was grounding to obtain? | the **poem** |

Both rates are legitimate; only the first describes the teacher, and it is the one carried into the report. The third is the most interesting at the tail — a poem needing several attempts has a genuinely high individual hallucination probability rather than bad luck, and poems that never ground at all are the closest thing here to *"text this teacher cannot reliably quote"*.

The first attempt's verdict is written at generation time and never overwritten, which is what makes the first row recoverable at all. Recomputing it later is impossible: the evidence has been replaced.

In [14]:
# The teacher's first draw, scored from the best evidence available per record:
# re-checked with the CURRENT checker where the stored text is genuinely the
# first attempt, and from the recorded flag where resampling inside a run
# replaced it. See teacher_hallucination_rate for why the two must be mixed.
by_id = {p["poem_id"]: p for p in raw_poems}
stored = [r for r in interpretations if r["poem_id"] in by_id]

rate, ci, n = data_filter.teacher_hallucination_rate(raw_poems)
survivors, _ = data_filter.hallucination_rate(stored, raw_poems,
                                              first_attempt_only=False)
attempts = data_filter.attempt_distribution(stored, raw_poems)

print(f"n = {n}\n")
print("hallucination rate, by measurement basis")
print(f"  first attempt (the teacher)     {rate:.2%}")
print(f"    95% CI (Wilson)               {ci[0]:.2%} - {ci[1]:.2%}")
print(f"  stored corpus (the survivors)   {survivors:.2%}")
print(f"    understates the teacher by    "
      f"{(rate - survivors) * 100:.2f} points")
print()
print("how hard grounding was to obtain")
print(f"  attempts per poem               {attempts['by_attempts']}")
print(f"  resampled at least once         {attempts['resampled']}")
print(f"  never grounded                  {attempts['ungrounded']}  "
      f"({attempts['ungrounded_rate']:.2%})")

n = 2550

hallucination rate, by measurement basis
  first attempt (the teacher)     4.27%
    95% CI (Wilson)               3.56% - 5.13%
  stored corpus (the survivors)   0.27%
    understates the teacher by    4.00 points

how hard grounding was to obtain
  attempts per poem               {1: 2369, 2: 123, 3: 17, 4: 25, 5: 5, 6: 3, 9: 1, 10: 7}
  resampled at least once         181
  never grounded                  7  (0.27%)


## What carries forward

This notebook produced the corpus and characterised the **teacher**. It made no
claim about what the corpus looks like — that is `02_eda.ipynb`, which loads the
same cached artifacts and describes them.

| Artifact | Where it goes |
|---|---|
| `data/poems.jsonl` — raw fetch, never re-fetched | cleaned at load time, everywhere downstream |
| `data/interpretations.jsonl` — append-only, resumable | the training targets |
| the funnel counts | Figure 1, and the data-quality section of the report |
| the teacher hallucination rate | the reference every later grounding number is read against |

The last row is the one to carry in mind. A student that quotes as accurately
as its teacher has learned what it was shown; one that quotes less accurately
has learned the format and dropped the substance — which is precisely the
distinction this project exists to measure.